<a href="https://colab.research.google.com/github/jbbb2/cam_ilt/blob/main/gen_train_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install fastmri
!pip install pygrappa
import fastmri as fmri
from fastmri.data import transforms
import pygrappa
import h5py
import subprocess
import glob
import os

In [ ]:
def extract_acs(kspace, n_acs):
    width = kspace.shape[2]
    center = width // 2
    return kspace[:,:,center - n_acs // 2:center + n_acs // 2]

In [ ]:
data_path = '/content/drive/MyDrive/colab_projects/data_image_recon/train_slices'
h5_path = '/content/drive/MyDrive/colab_projects/data_image_recon/train_volumes'
h5s = sorted(glob.glob(os.path.join(h5_path, 'file*.h5')))

In [ ]:
h5_path = '/content/drive/MyDrive/colab_projects/data_image_recon/train/multicoil_train'
h5s = sorted(glob.glob(os.path.join(h5_path, 'file*.h5')))

for file in h5s:
    print(file)

In [ ]:
for file in h5s:
    h5_file =  h5py.File(file)
    kspace = h5_file['kspace']

    for ii in range(kspace.shape[0]):
        slice_kspace = kspace[ii,:,:,:]
        kspace_tensor = fmri.data.transforms.to_tensor(slice_kspace)
        image = fmri.rss(fmri.complex_abs(fmri.ifft2c(kspace_tensor)), dim=0)
        mask_func = fmri.data.subsample.EquiSpacedMaskFunc([0.02], [8])
        kspace_subsampled_equi, _, n_acs = fmri.data.transforms.apply_mask(
            kspace_tensor, mask_func)
        image_subsampled_equi = fmri.rss(fmri.complex_abs(fmri.ifft2c(
            kspace_subsampled_equi)), dim=0)
        acs_lines = extract_acs(slice_kspace, n_acs)
        grappa_recon = pygrappa.grappa(fmri.data.transforms.tensor_to_complex_np(
            kspace_subsampled_equi), acs_lines, coil_axis=0, kernel_size=(3,3))
        grappa_recon_tensor = fmri.data.transforms.to_tensor(grappa_recon)
        grappa_image = fmri.rss(fmri.complex_abs(fmri.ifft2c(grappa_recon_tensor)), dim=0)

        with h5py.File(os.path.join(data_path, 'slice' + str(ii) + '_' +
                                    os.path.basename(file)), 'w') as new_h5:
            new_h5.create_dataset('input', data = grappa_image, compression='gzip')
            new_h5.create_dataset('output', data = image, compression='gzip')

    subprocess.run(['rm', '-f', data_path + file])